In [30]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
# create a session to reuse the underlying connection
session = requests.Session()

#
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
})

print("setup complete")

setup complete


In [31]:
def scrape_recipe(url):
    try:
        out = []
        headings = ["Name", "Ingredients", "Seasoning", "Marinade", "Sauce Mix", "Procedure"]
        headings_set = set(["Name", "Ingredients", "Seasoning", "Marinade", "Sauce Mix", "Procedure"])
        
        response = session.get(url, timeout=10)
        
        # This will raise an error if the site returns a 404 or 500 code
        response.raise_for_status() 
        
        soup = BeautifulSoup(response.text, 'html.parser')
        #print(soup)
        #https://usa.lkk.com/en/recipes/{recipe_path}
        temp_dict = {"Name": url.split('/')[-1]}
        
        
        boxes = soup.find_all('div', class_='side-box')   

        
        
        for box in boxes:
            heading = box.find('h3')
            
            if(heading):
                #print(heading.text)

                list_items = box.find_all("li")
                
                
                list_items = [x.text for x in list_items]
                #print(list_items)
                temp_dict[heading.text] = list_items

        
        steps = soup.find_all('li', class_='step-box')

        steps = [x.text for x in steps]
        temp_dict["Procedure"] = steps
            
        
        for heading in headings:
            if heading in temp_dict.keys():
                
                out.append(temp_dict[heading])
            else:
                out.append([])
        #print(out)
        # success
        print(f"Successfully scraped: {url}")
        
        return out
    except requests.exceptions.RequestException as e:
        print(f"Failed to scrape {url}: {e}")

In [32]:
def scrape_page_recipes(page_num):

    data = []
    
    url = f'https://usa.lkk.com/en/recipes?page={page_num}'
    
    print(f"Fetching main page: {url}")
    response = session.get(url, timeout=10)
    soup = BeautifulSoup(response.text, 'html.parser')

    recipes = soup.find_all('div', class_='recipe-item')

    recipe_set = set()
    
    
    for recipe in recipes:

        if recipe in recipe_set:
            continue
        
        link_element = recipe.find('a')
        
        if link_element and 'href' in link_element.attrs:

            
            recipe_path = link_element['href']
            
            # Construct the full URL
            full_url = f"https://usa.lkk.com{recipe_path}"
            
            
            data.append(scrape_recipe(full_url))
            
            
            time.sleep(2) 
        recipe_set.add(recipe)
    return data

In [33]:
def scrape(pages):
    data = []
    
    for page in range(1, pages + 1):
        data.extend(scrape_page_recipes(page))
    
    return pd.DataFrame(data, columns=["Name", "Ingredients", "Seasoning", "Marinade", "Sauce Mix", "Procedure"])

In [34]:

pages = 1
df = scrape(pages)
print(df.head())

df.to_csv('lkk_recipes.csv', index=False, encoding='utf-8')
print("data saved to lkk_recipes.csv!")

Fetching main page: https://usa.lkk.com/en/recipes?page=1
Successfully scraped: https://usa.lkk.com/en/recipes/air-fryer-xo-fish-balls-in-honey
Successfully scraped: https://usa.lkk.com/en/recipes/beef-and-broccoli-stir-fry
Successfully scraped: https://usa.lkk.com/en/recipes/beef-and-broccoli-stir-fry
Successfully scraped: https://usa.lkk.com/en/recipes/loaded-breakfast-hashbrown-waffles-recipe
Successfully scraped: https://usa.lkk.com/en/recipes/frozen-hot-pot-meal-prep-recipe
Successfully scraped: https://usa.lkk.com/en/recipes/buffalo-chicken-rangoons-recipe
Successfully scraped: https://usa.lkk.com/en/recipes/lomo-saltado-burrito-recipe
Successfully scraped: https://usa.lkk.com/en/recipes/ebi-katsu-curry-sando-shrimp-katsu-recipe
Successfully scraped: https://usa.lkk.com/en/recipes/creamy-mapo-tofu-udon-recipe
Successfully scraped: https://usa.lkk.com/en/recipes/spicy-yuzu-tamago-sando-recipe
Successfully scraped: https://usa.lkk.com/en/recipes/spicy-creamy-pasta-recipe-with-rose-